# Bronze Layer: Dimension Tables Ingestion

## Executive Summary

This notebook performs the **foundational ingestion** of critical business dimension data into the Bronze layer of our e-commerce analytics platform. Dimension tables are the descriptive backbone of data warehousing—they provide the "who, what, when, and where" context that transforms raw transactional data into meaningful business intelligence.

---

## Business Context: Why Dimensions Matter

### The Power of Dimensional Data

While fact tables record business events (orders, payments, shipments), **dimension tables provide the context** that makes those events interpretable and actionable:

**For Business Analysts:**
- **Product dimensions** enable analysis by brand, category, color, size, and material—answering "Which product lines are most profitable?"
- **Customer dimensions** support geographic segmentation and demographic analysis—answering "Where are our highest-value customers?"
- **Date dimensions** power time-series analysis, seasonality detection, and fiscal reporting—answering "How do Q4 sales compare to Q3?"

**For Marketing Teams:**
- Brand and category hierarchies inform **campaign targeting** and **promotional strategies**
- Customer geographic data enables **regional marketing optimization** and **localized offers**
- Product attributes support **merchandising decisions** and **assortment planning**

**For Data Science & ML:**
- Rich dimensional features are the **fuel for predictive models** (customer churn, product recommendations, demand forecasting)
- Slowly Changing Dimensions (SCDs) preserve **historical context** for accurate trend analysis
- Dimensional relationships enable **collaborative filtering** and **market basket analysis**

---

## What This Notebook Accomplishes

### Bronze Layer Mission: Preserve Raw Truth

The Bronze layer captures data **exactly as received** from source systems, acting as:
- 🛡️ **Immutable audit trail** for regulatory compliance and data recovery
- 🔄 **Reprocessing foundation** when business logic evolves
- 📜 **Historical archive** preserving the complete lifecycle of business entities

### Dimension Tables Ingested

This notebook processes **five critical dimension tables** from raw CSV files into Delta Lake tables:

#### 1. 🏷️ **Brands** (`brz_brand`)
**Business Value:** Tracks manufacturer and brand identities  
**Key Attributes:** Brand code (unique identifier), brand name, category association  
**Use Cases:**
- Brand performance analysis and competitive benchmarking
- Private label vs. national brand profitability comparison
- Supplier relationship management and contract negotiations

**Schema:**
- `brand_code` (String, PK) - Unique brand identifier
- `brand_name` (String) - Brand display name
- `category_code` (String, FK) - Links to category dimension
- `_source_file` (String) - Audit trail: originating file path
- `ingested_at` (Timestamp) - Audit trail: ingestion timestamp

---

#### 2. 📂 **Categories** (`brz_category`)
**Business Value:** Defines product taxonomy and merchandising hierarchy  
**Key Attributes:** Category code, category name  
**Use Cases:**
- Category performance dashboards and sales mix analysis
- Cross-category upselling opportunities ("Customers who bought X also viewed Y")
- Inventory planning and seasonal demand forecasting by category

**Schema:**
- `category_code` (String, PK) - Unique category identifier
- `category_name` (String) - Category display name
- `_source_file` (String) - Audit trail: originating file path
- `_ingested_at` (Timestamp) - Audit trail: ingestion timestamp

---

#### 3. 📦 **Products** (`brz_products`)
**Business Value:** Central catalog of all sellable items with physical and logical attributes  
**Key Attributes:** Product ID, SKU, category, brand, physical specs (weight, dimensions), ratings  
**Use Cases:**
- Product lifecycle management (introduction, growth, maturity, decline)
- Shipping cost optimization based on weight/dimensions
- Customer review sentiment correlation with product attributes
- Dynamic pricing based on brand, category, and rating

**Schema:**
- `product_id` (String, PK) - Unique product identifier
- `sku` (String) - Stock Keeping Unit (alternative identifier)
- `category_code` (String, FK) - Links to category dimension
- `brand_code` (String, FK) - Links to brand dimension
- `color`, `size`, `material` (String) - Physical product attributes
- `weight_grams`, `length_cm` (String) - **Note:** String type to handle data anomalies in source (e.g., "N/A", mixed units)
- `width_cm`, `height_cm` (Float) - Physical dimensions
- `rating_count` (Integer) - Number of customer reviews
- `file_name` (String) - Audit trail: originating file path
- `ingest_timestamp` (Timestamp) - Audit trail: ingestion timestamp

**Data Quality Note:** Weight and length captured as strings to accommodate upstream data inconsistencies—cleansing happens in Silver layer.

---

#### 4. 👥 **Customers** (`brz_customers`)
**Business Value:** Master data for customer identity and geographic distribution  
**Key Attributes:** Customer ID, phone, country, state  
**Use Cases:**
- Customer Lifetime Value (CLV) calculation and cohort analysis
- Geographic market penetration and regional sales performance
- Targeted marketing campaigns by location
- GDPR/CCPA compliance through customer data governance

**Schema:**
- `customer_id` (String, PK) - Unique customer identifier
- `phone` (String) - Customer contact (potentially PII—handle with care)
- `country_code`, `country`, `state` (String) - Geographic attributes
- `file_name` (String) - Audit trail: originating file path
- `ingest_timestamp` (Timestamp) - Audit trail: ingestion timestamp

**Privacy Consideration:** Contains PII (phone numbers)—access restricted via Unity Catalog RBAC.

---

#### 5. 📅 **Date/Calendar** (`brz_calendar`)
**Business Value:** Time dimension enabling temporal analysis and fiscal reporting  
**Key Attributes:** Date, year, day name, quarter, week of year  
**Use Cases:**
- Fiscal year and quarter reporting for finance teams
- Seasonality analysis (holiday sales spikes, back-to-school trends)
- Week-over-week and year-over-year growth calculations
- Workday vs. weekend sales pattern analysis

**Schema:**
- `date` (String) - Raw date value (cleansed in Silver layer)
- `year` (Integer) - Year component
- `day_name` (String) - Weekday name (e.g., "Monday")
- `quarter` (Integer) - Fiscal quarter (1-4)
- `week_of_year` (Integer) - **Note:** Source data may contain negative values (data quality issue handled downstream)
- `_source_file` (String) - Audit trail: originating file path
- `_ingested_at` (Timestamp) - Audit trail: ingestion timestamp

**Data Quality Note:** Negative week numbers and inconsistent date formats addressed in Silver layer transformations.

---

## Technical Approach: Bronze Layer Ingestion Patterns

### Ingestion Workflow (Applied to Each Dimension)

1. **Schema Definition**  
   - Explicit schema enforcement using PySpark `StructType`
   - **Business Benefit:** Prevents schema drift and data type mismatches that cause downstream failures

2. **Source Data Loading**  
   - Read from Unity Catalog Volumes (`/Volumes/ecommerce/source_data/raw/`)
   - CSV format with header row and comma delimiter
   - **Business Benefit:** Volumes provide governed, versioned storage with access control

3. **Metadata Enrichment**  
   - Add `_source_file` or `file_name` (lineage tracking)
   - Add `ingested_at` or `ingest_timestamp` (audit timestamp)
   - **Business Benefit:** Full traceability for compliance audits and data quality investigations

4. **Delta Lake Persistence**  
   - Write to Delta tables in `ecommerce.bronze` schema
   - `mode('overwrite')` for full refresh (Bronze typically replaces rather than merges)
   - `option('mergeSchema', 'true')` to handle schema evolution
   - **Business Benefit:** ACID transactions, time travel, and automatic schema evolution

---

## Data Quality Acknowledgments

### Known Data Anomalies (By Design)

The Bronze layer intentionally preserves raw data quality issues for transparency:

✅ **Products:**
- `weight_grams` and `length_cm` stored as strings due to inconsistent source formats ("N/A", mixed units)
- **Resolution:** Silver layer applies data cleansing and standardization

✅ **Date/Calendar:**
- Negative `week_of_year` values in source data
- Mixed case `day_name` values ("Monday" vs. "MONDAY")
- **Resolution:** Silver layer standardizes and validates temporal data

✅ **All Tables:**
- No deduplication applied at Bronze (preserves duplicate detection signals)
- No data quality rules enforced (enables retrospective quality analysis)

**Philosophy:** Bronze = Truth. Silver = Trusted Truth. Gold = Business-Ready Truth.

---

## Downstream Impact & Next Steps

Once these dimension tables are established in Bronze:

➡️ **Silver Layer Processing** - Apply data quality rules, standardize formats, deduplicate, and enrich with business logic  
➡️ **Gold Layer Analytics** - Create denormalized, aggregated dimension views for dashboards and ML features  
➡️ **Fact Table Joins** - Enable order, payment, and shipment facts to be enriched with dimensional context  
➡️ **Slowly Changing Dimensions (SCD)** - Track historical changes in customer addresses, product prices, and brand ownership

---

## Execution Prerequisites

✅ Unity Catalog `ecommerce` exists  
✅ Schema `ecommerce.bronze` exists  
✅ Source CSV files present in `/Volumes/ecommerce/source_data/raw/` with subdirectories: `brands/`, `category/`, `products/`, `customers/`, `date/`  
✅ User has `CREATE TABLE` permissions on `ecommerce.bronze` schema  
✅ Serverless compute available (auto-attached on execution)

---

## Notebook Execution Guide

1. **Run Cell 2:** Import required PySpark libraries (`StructType`, functions)
2. **Run Cell 3:** Set catalog name variable
3. **Run Cells 4-5:** Ingest **Brands** dimension
4. **Run Cell 7:** Ingest **Categories** dimension
5. **Run Cell 9:** Ingest **Products** dimension
6. **Run Cell 11:** Ingest **Customers** dimension
7. **Run Cell 13:** Ingest **Date/Calendar** dimension

**Validation:** Query each table after ingestion to verify record counts and schema correctness.

---

## Business Stakeholder Impact

| **Stakeholder** | **Enabled Capabilities** |
|----------------|-------------------------|
| **Executive Leadership** | Strategic KPIs by product category, brand performance scorecards, geographic market expansion analysis |
| **Product Management** | Product portfolio analysis, category performance tracking, brand positioning insights |
| **Marketing** | Customer segmentation by geography, brand campaign attribution, seasonal trend analysis |
| **Finance** | Fiscal period reporting, product profitability by category, regional revenue breakdowns |
| **Supply Chain** | Inventory turnover by brand/category, product dimension-based shipping optimization |
| **Data Science** | Feature engineering for recommendation engines, customer clustering, demand forecasting models |

---

**Ready to Execute:** Run the cells sequentially to populate the Bronze dimension tables. Monitor execution for schema validation errors or missing source files.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, FloatType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

#define schema for the data file

brand_schema = StructType([
    StructField('brand_code', StringType(), False),
    StructField('brand_name', StringType(), True),
    StructField('category_code', StringType(), True)
])

In [0]:
raw_data_path = '/Volumes/ecommerce/source_data/raw/brands/*.csv'

df = spark.read.option('header', 'true').option('delimiter', ',').schema(brand_schema).csv(raw_data_path)

#add metadata columns
df = df.withColumn('_source_file', F.col('_metadata.file_path')) \
       .withColumn('ingested_at', F.current_timestamp())

display(df.limit(5))

In [0]:
df.write.format('delta') \
    .mode('overwrite') \
    .option('mergeSchema', 'true') \
    .saveAsTable(f'{catalog_name}.bronze.brz_brand')

CATEGORY

In [0]:
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
])

# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/category/*.csv"

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(category_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at", F.current_timestamp()) \
               .withColumn("_source_file", F.col("_metadata.file_path"))


# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_category)
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_category")               

PRODUCT

In [0]:
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
])

# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/products/*.csv"

df = spark.read.option("header", "true").option("delimiter", ",").schema(products_schema).csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_products)
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_products")    

CUSTOMERS


In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

# Load data using the schema defined
raw_data_path ="/Volumes/ecommerce/source_data/raw/customers/*.csv"

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(customers_schema).csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_customers)
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_customers")      

DATE

In [0]:

# Define schema for the data file
date_schema = StructType([
    StructField("date", StringType(), True),           # Raw date in string format
    StructField("year", IntegerType(), True),          # Year
    StructField("day_name", StringType(), True),       # Day name (can be mixed case)
    StructField("quarter", IntegerType(), True),       # Quarter
    StructField("week_of_year", IntegerType(), True),  # Week of year (can be negative)
])

# Load data using the schema defined
raw_data_path = f"/Volumes/ecommerce/source_data/raw/date/*.csv" 

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(date_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at", F.current_timestamp()) \
               .withColumn("_source_file", F.col("_metadata.file_path"))


# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_calendar) 
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_calendar")               